# Fan-out join · **03 — The fan-out**

The baseline is the correct-grain query that passed the gate in notebook 02,
matching NHS England exactly by treatment function. Now I break it the way
it actually gets broken: with one extra join that looks harmless.

**The scenario.** Someone wants to slice waiting lists by ICB. The extract
has a provider's *host* ICB, but that isn't who pays for the care, so they
build a reference table of the ICBs each trust *mainly serves*: every ICB
(or NHS England commissioning hub) holding at least 20% of the trust's
incomplete pathways. That's a reasonable piece of reference data, made from
the same file. Then they join it to the fact table on `provider_code`,
because that's the only key the two share, and the report's treatment
function page keeps working.

The lookup is **one-to-many**: a border trust has two main ICBs. Every one
of that trust's fact rows now appears twice. Nothing raises an error.

I measure what that does, treatment function by treatment function.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import GOLD_DIR, STAR_DIR, TRUST_SUCCESSIONS  # noqa: E402

OUT = GOLD_DIR / "01_fan_out"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 220)

# The gate must have passed: I refuse to distort a baseline I haven't verified.
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all() and len(gate) == 53, "reconciliation gate has not passed; run 02 first"

fact = pd.read_parquet(STAR_DIR / "fact_incomplete_pathways.parquet")
dim_provider = pd.read_parquet(STAR_DIR / "dim_provider.parquet")
dim_commissioner = pd.read_parquet(STAR_DIR / "dim_commissioner.parquet")
tf = pd.read_parquet(STAR_DIR / "dim_treatment_function.parquet")
week = pd.read_parquet(STAR_DIR / "dim_week_band.parquet")
baseline = pd.read_csv(GOLD_DIR / "00_reconciliation" / "correct_grain_by_tf.csv")

## Step 1 — The lookup, built honestly from the star

Building it is itself a correct-grain query: fact → commissioner, which is
many-to-one. The lookup is fine as a table. It only becomes a problem
once it's joined to something on a key it isn't unique on.

In [2]:
THRESHOLD = 0.20

share = (fact.merge(dim_commissioner[["commissioner_code", "parent_code", "parent_name"]],
                    on="commissioner_code", validate="many_to_one")
             .groupby(["provider_code", "parent_code", "parent_name"], as_index=False).pathways.sum())
share["share_of_trust"] = share.pathways / share.groupby("provider_code").pathways.transform("sum")
assert share.groupby("provider_code").share_of_trust.sum().round(9).eq(1).all()

ref_main_icb = (share[share.share_of_trust >= THRESHOLD]
                .rename(columns={"parent_code": "icb_code", "parent_name": "icb_name"})
                [["provider_code", "icb_code", "icb_name", "share_of_trust"]]
                .sort_values(["provider_code", "share_of_trust"], ascending=[True, False], ignore_index=True))
ref_main_icb.to_csv(OUT / "ref_provider_main_icb.csv", index=False)

rows_per_provider = ref_main_icb.groupby("provider_code").size().reindex(dim_provider.provider_code, fill_value=0)
profile = rows_per_provider.value_counts().sort_index().rename_axis("lookup rows per provider").rename("providers")
print(f"lookup rows: {len(ref_main_icb)}  for {dim_provider.shape[0]} providers in the star")
profile.to_frame()

lookup rows: 621  for 532 providers in the star


,providers
lookup rows per provider,
0,2
1,443
2,83
3,4


Most trusts have one main ICB. The ones with two or three are the trusts
whose rows will be repeated. And there's a second problem in the same
table: a few providers have **no** ICB at 20% (their list is spread too
thinly), so an inner join drops them without saying so.

In [3]:
dropped = dim_provider.set_index("provider_code").loc[rows_per_provider[rows_per_provider == 0].index, ["provider_name"]]
dropped["pathways"] = fact.groupby("provider_code").pathways.sum().reindex(dropped.index)
dropped

,provider_name,pathways
provider_code,,
AAV,COMMUNITY HEALTH AND EYECARE LIMITED,4754
Q2Y5H,OVIVA UK LTD,11729


## Step 2 — The same query, with one more join

`by_treatment_function` is the gated query from notebook 02, unchanged.
The fan-out version is the same function with the lookup joined in first.
pandas' default `merge` is SQL's `INNER JOIN`: no cardinality check, no
warning, no error.

In [4]:
def by_treatment_function(f: pd.DataFrame) -> pd.DataFrame:
    x = (f.merge(week[["week_band", "within_18"]], on="week_band", validate="many_to_one")
          .merge(tf, on="tf_code", validate="many_to_one"))
    x["within_18_pathways"] = x.pathways.where(x.within_18, 0)
    g = x.groupby(["tf_code", "tf_name"], as_index=False)[["pathways", "within_18_pathways"]].sum()
    g["pct_within_18"] = 100 * g.within_18_pathways / g.pathways
    g["pct_over_18"] = 100 - g.pct_within_18
    return g

correct = by_treatment_function(fact)
# The baseline here must be the one the gate passed, to the pathway.
assert (correct.set_index("tf_code").pathways == baseline.set_index("tf_code").pathways).all()

fanned_rows = fact.merge(ref_main_icb[["provider_code", "icb_code"]], on="provider_code")   # no validate, no error
fanned = by_treatment_function(fanned_rows)
print(f"fact rows {len(fact):,} -> {len(fanned_rows):,} after the join; no exception raised")

fact rows 501,422 -> 564,592 after the join; no exception raised


The one argument that would have stopped it. `validate="many_to_one"` makes
pandas check the data at the moment of the join: that's validation. It's
the pandas equivalent of a test, not of a declaration.

In [5]:
try:
    fact.merge(ref_main_icb[["provider_code", "icb_code"]], on="provider_code", validate="many_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 provider_code
        A1U4J
          AAH
        ACG12
        ACG26
        ACG37 ...


## Step 3 — The gap, treatment function by treatment function

In [6]:
cmp = (correct[["tf_code", "tf_name", "pathways", "pct_over_18"]]
       .merge(fanned[["tf_code", "pathways", "pct_over_18"]], on="tf_code", suffixes=("_correct", "_fanout")))
cmp["gap_points"] = cmp.pct_over_18_fanout - cmp.pct_over_18_correct
cmp["denominator_inflation"] = cmp.pathways_fanout / cmp.pathways_correct
cmp = cmp.sort_values("gap_points", ascending=False, ignore_index=True)
cmp.to_csv(OUT / "fan_out_by_tf.csv", index=False)
cmp.round(3)

,tf_code,tf_name,pathways_correct,pct_over_18_correct,pathways_fanout,pct_over_18_fanout,gap_points,denominator_inflation
0,C_300,General Internal Medicine Service,28907,25.821,30794,26.508,0.688,1.065
1,C_400,Neurology Service,205335,37.633,238179,38.021,0.388,1.160
2,C_110,Trauma and Orthopaedic Service,843419,38.900,967860,39.278,0.379,1.148
3,C_120,Ear Nose and Throat Service,589316,43.231,651818,43.565,0.334,1.106
4,C_320,Cardiology Service,360799,32.113,401432,32.423,0.311,1.113
5,C_140,Oral Surgery Service,315652,46.183,345911,46.337,0.154,1.096
6,C_301,Gastroenterology Service,357430,30.323,390239,30.460,0.136,1.092
7,X04,Other - Paediatric Services,321302,34.032,349997,34.153,0.122,1.089
8,C_100,General Surgery Service,384427,36.822,429402,36.927,0.105,1.117
9,X02,Other - Medical Services,620378,29.894,676118,29.951,0.058,1.090


### Why the rate moves at all (and why it isn't "the denominator doubles")

If the join repeated **every** row the same number of times, numerator and
denominator would both scale by the same factor and the percentage would
not move. Rates are immune to uniform duplication. A fan-out distorts a
rate only because it repeats **some** rows, here the rows of trusts with
two or three main ICBs, so the result is a re-weighted average, tilted
towards those trusts. The rate moves up in a treatment function where the
repeated trusts wait longer than everyone else, and down where they wait
less. The direction depends on who happens to be repeated, which is why it
can't be eyeballed.

I check that explanation against the data: within each treatment function,
the fan-out rate must be exactly the pathway-weighted rate with each trust
weighted by its number of lookup rows.

In [7]:
pt = (fact.merge(week[["week_band", "within_18"]], on="week_band")
          .assign(w18=lambda d: d.pathways.where(d.within_18, 0))
          .groupby(["tf_code", "provider_code"], as_index=False)[["pathways", "w18"]].sum())
pt["k"] = pt.provider_code.map(rows_per_provider)
re_weighted = pt.assign(p=pt.pathways * pt.k, w=pt.w18 * pt.k).groupby("tf_code")[["p", "w"]].sum()
re_weighted_pct = 100 * (1 - re_weighted.w / re_weighted.p)
assert (re_weighted_pct - cmp.set_index("tf_code").pct_over_18_fanout).abs().max() < 1e-9
print("confirmed: the fan-out rate is exactly a re-weighting by lookup rows per trust")

confirmed: the fan-out rate is exactly a re-weighting by lookup rows per trust


## Step 4 — The explanatory table, on the real figures

In [8]:
eng = lambda g: (int(g.pathways.sum()), 100 * (1 - g.within_18_pathways.sum() / g.pathways.sum()))
n_c, pct_c = eng(correct)
n_f, pct_f = eng(fanned)
explain = pd.DataFrame({
    "": ["Rows returned", "Denominator (England pathways)", "England % over 18 weeks",
         "Specialties moved 0.5+ points", "Largest move", "Providers silently dropped",
         "Error raised", "Looks plausible"],
    "At pathway grain": [f"{len(fact):,} (each pathway once)", f"{n_c:,}", f"{pct_c:.1f}%",
                         "—", "—", "0", "—", "Yes"],
    "Through the fan-out": [f"{len(fanned_rows):,} (rows × main ICBs per trust)", f"{n_f:,} (×{n_f / n_c:.2f})",
                            f"{pct_f:.1f}%", f"{(cmp.gap_points.abs() >= 0.5).sum()} of 23",
                            f"{cmp.loc[cmp.gap_points.abs().idxmax(), 'tf_name'].removesuffix(' Service')} "
                            f"{cmp.loc[cmp.gap_points.abs().idxmax(), 'gap_points']:+.2f} pts".replace("-", "−"),
                            f"{len(dropped)} ({int(dropped.pathways.sum()):,} pathways)", "None", "Yes"],
})
explain.to_csv(OUT / "explanatory_table.csv", index=False)
explain

,,At pathway grain,Through the fan-out
0,Rows returned,"501,422 (each pathway once)","564,592 (rows × main ICBs per trust)"
1,Denominator (England pathways),"7,158,164","8,051,914 (×1.12)"
2,England % over 18 weeks,34.8%,34.7%
3,Specialties moved 0.5+ points,—,5 of 23
4,Largest move,—,Dermatology −0.75 pts
5,Providers silently dropped,0,"2 (16,483 pathways)"
6,Error raised,—,None
7,Looks plausible,Yes,Yes


## Step 5 — A contrast: the crude fan-out gets caught

The same join with a lookup of **every** ICB a trust serves (no threshold).
It's the same mistake, but this one doesn't survive the meeting: the
England total runs to hundreds of millions, which anyone would notice. The
dangerous fan-out is the modest one.

In [9]:
ref_all_icb = share[["provider_code", "parent_code"]]
crude_rows = fact.merge(ref_all_icb, on="provider_code")
crude = by_treatment_function(crude_rows)
n_x, pct_x = eng(crude)
contrast = pd.DataFrame([
    {"lookup": "none (correct grain)", "rows": len(fact), "england_pathways": n_c, "inflation": 1.0,
     "england_pct_over_18": pct_c, "max_abs_tf_gap_points": 0.0},
    {"lookup": f"main ICBs (>= {THRESHOLD:.0%} of trust list)", "rows": len(fanned_rows), "england_pathways": n_f,
     "inflation": n_f / n_c, "england_pct_over_18": pct_f, "max_abs_tf_gap_points": cmp.gap_points.abs().max()},
    {"lookup": "every ICB the trust serves", "rows": len(crude_rows), "england_pathways": n_x, "inflation": n_x / n_c,
     "england_pct_over_18": pct_x,
     "max_abs_tf_gap_points": (crude.set_index("tf_code").pct_over_18 - correct.set_index("tf_code").pct_over_18).abs().max()},
])
contrast.to_csv(OUT / "contrast.csv", index=False)
contrast.round(3)

,lookup,rows,england_pathways,inflation,england_pct_over_18,max_abs_tf_gap_points
0,none (correct grain),501422,7158164,1.000,34.794,0.000
1,main ICBs (>= 20% of trust list),564592,8051914,1.125,34.708,0.748
2,every ICB the trust serves,15779282,239428768,33.448,35.718,2.492


### How much does the threshold matter?

20% is my choice, so I show what nearby choices do. Each row is a
different but equally defensible "main ICB" lookup.

In [10]:
sens = []
for thr in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    lk = share.loc[share.share_of_trust >= thr, ["provider_code", "parent_code"]]
    g = by_treatment_function(fact.merge(lk, on="provider_code"))
    gap = g.set_index("tf_code").pct_over_18 - correct.set_index("tf_code").pct_over_18
    k = lk.groupby("provider_code").size()
    sens.append({"threshold": thr, "providers_repeated": int((k > 1).sum()),
                 "providers_dropped": int(len(dim_provider) - len(k)),
                 "england_inflation": g.pathways.sum() / n_c,
                 "england_pct_over_18": 100 * (1 - g.within_18_pathways.sum() / g.pathways.sum()),
                 "min_tf_gap": gap.min(), "max_tf_gap": gap.max(), "tfs_moved_0_5_plus": int((gap.abs() >= 0.5).sum())})
sens = pd.DataFrame(sens)
sens.to_csv(OUT / "threshold_sensitivity.csv", index=False)
sens.round(3)

,threshold,providers_repeated,providers_dropped,england_inflation,england_pct_over_18,min_tf_gap,max_tf_gap,tfs_moved_0_5_plus
0,0.10,174,0,1.351,34.360,-2.479,4.046,10
1,0.15,122,0,1.210,34.691,-1.620,2.912,9
2,0.20,87,2,1.125,34.708,-0.748,0.688,5
3,0.25,63,4,1.086,34.889,-1.890,0.816,4
4,0.30,51,4,1.073,34.898,-1.890,0.855,4
5,0.40,15,10,1.012,35.138,-1.655,2.226,5


### Does the fan-out change which specialties look worst?

A league table is what a review meeting actually reads.

In [11]:
rank = cmp.assign(rank_correct=cmp.pct_over_18_correct.rank(ascending=False).astype(int),
                  rank_fanout=cmp.pct_over_18_fanout.rank(ascending=False).astype(int))
rank["rank_change"] = rank.rank_correct - rank.rank_fanout
rank.to_csv(OUT / "rank_change.csv", index=False)
print(f"treatment functions whose rank changes: {(rank.rank_change != 0).sum()} of 23")
rank.loc[rank.rank_change != 0, ["tf_name", "pct_over_18_correct", "pct_over_18_fanout", "rank_correct", "rank_fanout"]] \
    .sort_values("rank_correct").round(2)

treatment functions whose rank changes: 6 of 23


,tf_name,pct_over_18_correct,pct_over_18_fanout,rank_correct,rank_fanout
16,Neurosurgical Service,38.04,37.66,6,7
1,Neurology Service,37.63,38.02,7,6
22,Dermatology Service,30.96,30.21,13,14
6,Gastroenterology Service,30.32,30.46,14,13
21,Cardiothoracic Surgery Service,26.07,25.33,18,20
0,General Internal Medicine Service,25.82,26.51,20,18


### Where the fan-out does nothing at all

Joined on `provider_code`, the fan-out repeats all of a trust's rows the
same number of times. So **any figure for a single trust is unchanged**:
its rate is immune, and only its count inflates. The distortion appears
only when trusts are added together. I check that, because it's the
reason a trust-level sense check wouldn't catch this.

In [12]:
def by_provider_tf(f):
    x = f.merge(week[["week_band", "within_18"]], on="week_band")
    x["w"] = x.pathways.where(x.within_18, 0)
    g = x.groupby(["provider_code", "tf_code"])[["pathways", "w"]].sum()
    return 100 * (1 - g.w / g.pathways)
a, b = by_provider_tf(fact), by_provider_tf(fanned_rows)
common = a.index.intersection(b.index)
assert (a.loc[common] - b.loc[common]).abs().max() < 1e-9
print(f"{len(common):,} provider x treatment function rates: identical through the fan-out")

3,502 provider x treatment function rates: identical through the fan-out


## Step 6 — The merger version, for comparison

The other fan-out NHS data invites is a provider history table with two
rows for a merged trust: the survivor's own row and the row it inherited.
The only merger in April 2025 - August 2026 is North Bristol (RVJ) into
Bristol NHS FT (RA7) on 1 July 2026, so a successor lookup joined on the
surviving code repeats exactly one trust. I measure it so the choice of
headline example is made on numbers, not taste.

In [13]:
succ = pd.DataFrame(
    [(new, new) for _, new, _ in TRUST_SUCCESSIONS] + [(new, old) for old, new, _ in TRUST_SUCCESSIONS],
    columns=["provider_code", "historic_code"])
print("RVJ submitting in August 2026:", "RVJ" in set(fact.provider_code))
merger_rows = fact.merge(succ, on="provider_code", how="left")    # unmatched providers keep one row
merger = by_treatment_function(merger_rows)
mg = (merger.set_index("tf_name").pct_over_18 - correct.set_index("tf_name").pct_over_18).rename("gap_points")
print(f"rows {len(fact):,} -> {len(merger_rows):,}; England pathways ×{merger.pathways.sum() / n_c:.4f}; "
      f"largest treatment-function move {mg.abs().max():.2f} points; moves of 0.5+: {(mg.abs() >= 0.5).sum()}")
mg.sort_values(key=abs, ascending=False).head(5).round(3).to_frame()

RVJ submitting in August 2026: False


rows 501,422 -> 506,302; England pathways ×1.0134; largest treatment-function move 0.33 points; moves of 0.5+: 0


,gap_points
tf_name,
Neurosurgical Service,-0.334
Plastic Surgery Service,-0.239
Other - Paediatric Services,0.229
Other - Surgical Services,-0.178
Dermatology Service,-0.157


## Step 7 — Declared is not validated

The vendors' answer to this is to let you **declare** the cardinality: a
relationship constraint on a metric view, a cardinality setting on a
semantic-model relationship. Below, I declare the lookup's relationship
the way the analyst in this scenario believed it was: one ICB per trust.
The declaration is a record of intent. It doesn't read a single row.
Checking it is a separate act, and the data contradicts it.

In [14]:
DECLARED = {("fact_incomplete_pathways", "ref_provider_main_icb"): {"on": "provider_code", "cardinality": "many_to_one"}}

def check_declared(left: pd.DataFrame, right: pd.DataFrame, on: str, cardinality: str) -> dict:
    """What the declaration claims vs what the data says. Only this function looks at rows."""
    right_unique = right[on].is_unique
    left_keys = set(left[on]); right_keys = set(right[on])
    return {"declared": cardinality,
            "right side unique on key": right_unique,
            "max right rows per key": int(right.groupby(on).size().max()),
            "left keys with no match": len(left_keys - right_keys),
            "holds": (cardinality == "many_to_one" and right_unique)}

check_declared(fact, ref_main_icb, **DECLARED[("fact_incomplete_pathways", "ref_provider_main_icb")])

{'declared': 'many_to_one',
 'right side unique on key': False,
 'max right rows per key': 3,
 'left keys with no match': 2,
 'holds': False}